# 08 — Momentos de predicción y lags del rinde

Dos extensiones que pide la consigna, como **ablations** sobre el mejor modelo:

1. **Dos momentos del calendario agrícola** (`momento` en `datos.build_reg_dataset`):
   - `pre_siembra` — solo lo conocible antes de sembrar (ONI + humedad de suelo de
     invierno + estructurales). Sirve para decisiones tempranas.
   - `pre_cosecha` — clima y NDVI hasta febrero (sin marzo).
   - `full` — toda la campaña Sep–Mar (referencia).
2. **Lags del rinde** (`use_lags`): rinde de 1–k campañas previas + media móvil,
   todo con `shift` (nunca el año actual → sin leakage).

In [ ]:
import sys, os, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.float_format', lambda v: f'{v:,.3f}')
import datos, evaluacion as ev, json
from modelos import XGBoostRegressor
CULTIVO = 'soja'
BEST = json.load(open('retuning_cv_honesta.json', encoding='utf-8')).get(CULTIVO, {}).get('xgb', {}).get('best_params', {}) if os.path.exists('retuning_cv_honesta.json') else {}
def evalua(momento='full', use_lags=0, use_agro=None):
    if use_agro is None: use_agro = (momento == 'full')   # agro llega a marzo
    d = datos.prepare(CULTIVO, momento=momento, use_lags=use_lags, use_agro=use_agro, enc_smooth=10.0)
    m = XGBoostRegressor(**BEST, random_state=42).fit(d.X_train, d.y_train)
    p = m.predict(d.X_test)
    r = ev.metricas(d.y_test, p)
    r['skill'] = ev.skill_score(d.y_test, p, ev.pred_media_depto(d))
    r['n_feats'] = len(d.feature_cols)
    return r, d, p

## 1. Los tres momentos de predicción

In [ ]:
filas = []
res = {}
for mom in ['pre_siembra', 'pre_cosecha', 'full']:
    r, d, p = evalua(momento=mom)
    res[mom] = (d, p)
    filas.append({'momento': mom, 'n_feats': r['n_feats'], 'RMSE': r['rmse'],
                  'R2': r['r2'], 'sMAPE': r['smape'], 'skill': r['skill']})
tabla_mom = pd.DataFrame(filas)
tabla_mom

Cuanto más tarde en la campaña se predice, más información climática observada hay
→ mejor debería ser. `pre_siembra` es el caso más difícil (casi sin clima del año);
si aun así le gana a la climatología, hay señal anticipable (ENSO, humedad de suelo).

In [ ]:
# ¿La diferencia full vs pre_cosecha es significativa? (mismas filas de test)
d_full, p_full = res['full']; d_pc, p_pc = res['pre_cosecha']
dm = ev.diebold_mariano(d_full.y_test, p_full, p_pc, loss='se')
print(f"Diebold-Mariano full vs pre_cosecha: DM={dm['dm']:.2f}  p={dm['p_value']:.3f}  "
      f"-> {'diferencia significativa' if dm['p_value'] < 0.05 else 'sin diferencia significativa'}")

## 2. Lags del rinde (features autorregresivas)

In [ ]:
filas = []
for k in [0, 1, 3, 5]:
    r, _, _ = evalua(momento='full', use_lags=k, use_agro=True)
    filas.append({'lags': k, 'n_feats': r['n_feats'], 'RMSE': r['rmse'],
                  'R2': r['r2'], 'skill': r['skill']})
tabla_lags = pd.DataFrame(filas)
tabla_lags

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(tabla_mom['momento'], tabla_mom['R2'], color='#4C72B0')
ax[0].set_title('R² por momento de predicción'); ax[0].axhline(0, color='0.6', lw=0.8)
ax[1].plot(tabla_lags['lags'], tabla_lags['R2'], marker='o', color='#55A868')
ax[1].set_title('R² vs. nº de lags del rinde'); ax[1].set_xlabel('lags')
plt.tight_layout(); plt.show()

## Conclusión

- **Momentos:** `full` y `pre_cosecha` rinden parecido (el clima de marzo aporta
  poco una vez que están sep–feb y el NDVI); el Diebold–Mariano dice si la diferencia
  es real. `pre_siembra` es mucho más difícil — su skill vs. climatología mide cuánto
  se puede **anticipar** antes de sembrar (ENSO + humedad de suelo).
- **Lags:** agregar el rinde de campañas previas aporta señal autorregresiva
  (persistencia depto), complementaria al `depto_enc`. El punto óptimo de k se ve en
  la curva; más lags achican el train (se pierden las primeras campañas de cada serie).